# 09 — The trap table

**Reference, not taught.** Nobody runs this in a workshop. It is the thing people keep.

Every entry here cost real time, and **not one of them is in any documentation**. That
is the reason this workshop exists as a workshop and not a link list: the APIs are
documented, the *failure modes* are not.

**39 traps.** 38 were reproduced against
live services while writing these notebooks; the rest are marked accordingly.

## How to use this

When something is wrong and you do not know why, come here and match on the **symptom**,
not on the cause. Symptoms are what you have; causes are what you are trying to find.

The `live` column means the failure was reproduced and the fix verified. `documented`
means it is a known property of the service rather than something re-derived here.

In [1]:
import sys
from pathlib import Path

# notebooks/ holds _fetch.py and _sources.py; src/ holds the project's own helpers.
for p in (Path.cwd(), Path.cwd() / ".." / "src"):
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p.resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import _fetch
import _sources as S

# A real requests.Session, with a disk cache. Everything you use below -- .get,
# .text, .content, .json, .status_code, .raise_for_status, params= -- is
# ordinary `requests`. The cache is invisible at the call site.
http = _fetch.session()

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

_fetch.OFFLINE and print("offline mode: serving prefetched responses")

offline mode: serving prefetched responses


In [2]:
# The same table, machine-readable -- so you can grep it.
traps = pd.DataFrame(
    [
        ('04 access policy', "403 and 404 both look like 'not allowed'", 'you go looking for an API key to fix a bucket name that does not exist; S3 answers NoSuchBucket (404) for a wrong name and AccessDenied (403) for a real one, and assert resp.ok collapses the difference', 'read the error body before authenticating: 404 needs a new name, 403 needs a key', 'live'),
        ('04 access policy', 'a public-looking bucket is not free to fetch', "landsat-pds reads as open, then answers 403 'Anonymous users cannot invoke requests against Requester Pays buckets. Please authenticate.' -- the owner pays egress, so the bytes are billable to whoever pulls them", 'requests pays nothing; Athena or Spectrum over an open bucket does', 'live'),
        ('04 access policy', 'free public data still refuses an anonymous request', "POST to the public-data project returns 401 'Request is missing required authentication credential' before the query is even looked at", 'a Google account and project; the sandbox needs no credit card, and the first 1 TiB of query processing per month is free', 'live'),
        ('01/02 ERDDAP', 'CSV has a names row AND a units row', 'first row of strings; off-by-one on every row after', 'pd.read_csv(..., skiprows=[1])', 'live'),
        ('01 ERDDAP', 'index expression is part of the parameter NAME, not its value', "500 destinationVariableName=... wasn't found -- from either mistake", 'hand-build the query string; params= cannot express it', 'live'),
        ('01 ERDDAP', 'curl reads [ ] as a glob pattern', 'exit code 3, URL malformat, and NO message under -s', 'curl -g / --globoff', 'live'),
        ('02 ERDDAP', '.time=first / .lat=first are a no-op', 'byte-identical response; you get 520x the data you asked for', 'shrink the lat/lon box instead', 'live'),
        ('02 ERDDAP', 'constraint variables (&time=) rejected in griddap', "400: '&' must be followed by a .griddap server variable", 'use the index form only', 'live'),
        ('02 ERDDAP', 'strides in the index [(0):(1):(29)] rejected', '400: For variable=... axis#0=time Constraint=...', 'no strides exist; shrink the box', 'live'),
        ('02 ERDDAP', 'grid coordinates are float32', "latitude.max() = 36.81999969 is 'outside' the box you asked for", 'compare with a 1e-3 tolerance, never exactly', 'live'),
        ('02 ERDDAP', 'NaN fill is skipped by reductions; -999.0 fill is not', 'mean silently wrong, no error', 'check the min, and mask explicitly', 'live'),
        ('02 ERDDAP', 'large CSV responses are paged', 'page 2 without page 1 returns empty with HTTP 200', 'request pages in order', 'documented'),
        ('01-08 any', 'AAAA records but no IPv6 route', 'requests ~100x slower than curl; every call takes exactly the timeout', 'force AF_INET (the workshop helper does this)', 'live'),
        ('03 GCS', 'the data/ level is mandatory when FETCHING', '404 with <Code>NoSuchKey</Code> -- reads as a permissions problem', 'include /data/ in the object key', 'live'),
        ('03 GCS', 'error bodies are XML even from the JSON API', 'r.json() raises JSONDecodeError, hiding the real message', 'raise_for_status() then read r.text', 'live'),
        ('03 GCS', 'three capitalisations in one path', 'directory all-lower, file title-case, trailing unit lower: TOL_1h not TOL_1H', 'copy a real listing, do not construct names by upper()', 'live'),
        ('03 GCS', 'aws s3 honours ~/.aws/config', 'commands silently redirected to a local MinIO on localhost:9000', 'use the GCS JSON API -- no profile, no region, no credentials', 'live'),
        ('03 GCS', 'same recording at four resolutions', 'psd_1h is ~456 MB against tol_1h at ~0.7 MB', 'list first, read the size, then download', 'live'),
        ('05 Argo', 'N_PARAM is in ds.sizes but unused by TEMP/PSAL/PRES', 'ValueError on isel, or a silently wrong selection if you ignore it', 'check .shape before indexing', 'live'),
        ('05 Argo', 'QC flags are bytes in _prof.nc, int8 in per-cycle files', '(qc == 1) is silently all-False', 'decode to int first; count the codes, do not use .all()', 'live'),
        ('05 Argo', 'no index and no search API over 4,261 floats', '/index/, /dac/index/, _prof_index.txt all 404', 'plan for screening, or use a source that has an index', 'live'),
        ('05 Argo', "no small file holds a float's position", '_meta.nc (35 KB) and _tech.nc (2.8 MB) both lack LATITUDE', 'position is in _prof.nc; screening costs 689 KB per candidate', 'live'),
        ('05 Argo', 'argopy is broken against erddapy 3.x', 'imports the removed _quote_string_constraints', 'fetch GDAC netCDF directly; do not downgrade xarray', 'live'),
        ('07 NDBC', 'line 0 is names, line 1 is units, data starts at line 2', 'columns named #yr, mo, dy; KeyError much later', 'parse lines[0], slice from lines[2]', 'live'),
        ('07 NDBC', 'resolution is not uniform within a year file', '7,835 rows for 2019, not 8,760; file starts 5 January', 'resample explicitly; never assume hourly', 'live'),
        ('07 NDBC', 'sentinels are per column, and 99 is a valid bearing', 'a blanket >= 99 filter deletes real easterly winds', 'mask per column: 99 generally, 999 for WDIR/MWD', 'live'),
        ('07 NDBC', 'RangeIndex Series into DataFrame(index=DatetimeIndex)', 'aligns on index; every column silently NaN', 'pass .to_numpy() so values are positional', 'live'),
        ('07 NDBC', 'wind direction is FROM, and bearings wrap', 'mean(350, 10) = 180, the exact opposite of 0; 79.6 deg error over 2019', 'average the u/v vectors, then atan2', 'live'),
        ('08 gsw', 'Conservative Temperature is degrees C, not Kelvin', 'returns nan with only a RuntimeWarning; propagates silently', 'pass degC, or convert with gsw.CT_from_t', 'live'),
        ('08 gsw', 'SA is not SP', 'SA_from_SP needs pressure AND position; ~0.17 g/kg apart', 'use gsw.SA_from_SP(sp, p, lon, lat)', 'live'),
        ('08 gsw', 'dc/dp is ~17 m/s per 1000 dbar', 'a per-dbar reading of a per-1000 quantity is 100x wrong', 'state the units of whatever you are comparing against', 'live'),
        ('06 Copernicus', 'describe with no filter returns a 170 MB catalogue', "multi-minute hang on a 'describe' call", 'always pass --dataset-id and a filter', 'live'),
        ('06 Copernicus', 'the flag is --end-datetime', 'not --stop-datetime, which is rejected', 'check --help; the CLI is not consistent across tools', 'live'),
        ('06 Copernicus', 'advertised variable count is reported as 0', 'a correct dataset that looks empty', 'do not trust the summary; request variables explicitly', 'live'),
        ('06 Copernicus', 'the human Product ID and the CLI dataset id differ', "'Dataset not found' for GLOBAL_MULTIYEAR_PHY_001_030, which is live", 'the CLI wants the CMEMS product code cmems_mod_glo_phy_my_0.083deg_P1D-m', 'live'),
        ('06 Copernicus', 'the error tells you to check the dataset id', 'the id looks authoritative, so you start editing a correct one', 'diff the failing argv against a working one before theorising', 'live'),
        ('06 Copernicus', 'login env vars are COPERNICUSMARINE_SERVICE_USERNAME/PASSWORD', 'COPERNICUS_USERNAME / COPERNICUS_PASSWORD are silently ignored', 'read `copernicusmarine login --help` for the real names', 'live'),
        ('09 SQL', "Postgres names every avg() result 'avg'", 'three averages in one SELECT produce duplicate column names', 'alias every aggregate explicitly', 'live'),
        ('09 SQL', "int(31.5) uses banker's rounding", '31.5 Hz becomes band_32hz, 62.5 would become band_62hz', 'state the rule, and test the boundary case', 'live'),
    ],
    # NOT "where": DataFrame.where is a method, so traps.where silently resolves
    # to the method rather than the column. A column name that shadows an API is
    # a trap in its own right, which is a slightly embarrassing way to make this
    # table.
    columns=["source", "what_breaks", "symptom", "fix", "verified"],
)
traps.index.name = "id"
print(f"  {len(traps)} traps")
print()
print("  by source:")
for src, n in traps.source.value_counts().items():
    print(f"    {src:22} {n}")
print()
print("  verified live:", int((traps.verified == "live").sum()))
traps.to_csv("_traps.csv", index=False)
print("  written to _traps.csv")

  39 traps

  by source:
    02 ERDDAP              6
    06 Copernicus          6
    03 GCS                 5
    05 Argo                5
    07 NDBC                5
    04 access policy       3
    08 gsw                 3
    01 ERDDAP              2
    09 SQL                 2
    01/02 ERDDAP           1
    01-08 any              1

  verified live: 38
  written to _traps.csv


## The full list

| # | where | what breaks | symptom | fix | |
|---|---|---|---|---|---|
| 1 | 04 access policy | you go looking for an API key to fix a bucket name that does not exist; S3 answers NoSuchBucket (404) for a wrong name and AccessDenied (403) for a real one, and assert resp.ok collapses the difference | read the error body before authenticating: 404 needs a new name, 403 needs a key | live |
| 2 | 04 access policy | landsat-pds reads as open, then answers 403 'Anonymous users cannot invoke requests against Requester Pays buckets. Please authenticate.' -- the owner pays egress, so the bytes are billable to whoever pulls them | requests pays nothing; Athena or Spectrum over an open bucket does | live |
| 3 | 04 access policy | POST to the public-data project returns 401 'Request is missing required authentication credential' before the query is even looked at | a Google account and project; the sandbox needs no credit card, and the first 1 TiB of query processing per month is free | live |
| 4 | 01/02 ERDDAP | first row of strings; off-by-one on every row after | pd.read_csv(..., skiprows=[1]) | live |
| 5 | 01 ERDDAP | 500 destinationVariableName=... wasn't found -- from either mistake | hand-build the query string; params= cannot express it | live |
| 6 | 01 ERDDAP | exit code 3, URL malformat, and NO message under -s | curl -g / --globoff | live |
| 7 | 02 ERDDAP | byte-identical response; you get 520x the data you asked for | shrink the lat/lon box instead | live |
| 8 | 02 ERDDAP | 400: '&' must be followed by a .griddap server variable | use the index form only | live |
| 9 | 02 ERDDAP | 400: For variable=... axis#0=time Constraint=... | no strides exist; shrink the box | live |
| 10 | 02 ERDDAP | latitude.max() = 36.81999969 is 'outside' the box you asked for | compare with a 1e-3 tolerance, never exactly | live |
| 11 | 02 ERDDAP | mean silently wrong, no error | check the min, and mask explicitly | live |
| 12 | 02 ERDDAP | page 2 without page 1 returns empty with HTTP 200 | request pages in order | documented |
| 13 | 01-08 any | requests ~100x slower than curl; every call takes exactly the timeout | force AF_INET (the workshop helper does this) | live |
| 14 | 03 GCS | 404 with <Code>NoSuchKey</Code> -- reads as a permissions problem | include /data/ in the object key | live |
| 15 | 03 GCS | r.json() raises JSONDecodeError, hiding the real message | raise_for_status() then read r.text | live |
| 16 | 03 GCS | directory all-lower, file title-case, trailing unit lower: TOL_1h not TOL_1H | copy a real listing, do not construct names by upper() | live |
| 17 | 03 GCS | commands silently redirected to a local MinIO on localhost:9000 | use the GCS JSON API -- no profile, no region, no credentials | live |
| 18 | 03 GCS | psd_1h is ~456 MB against tol_1h at ~0.7 MB | list first, read the size, then download | live |
| 19 | 05 Argo | ValueError on isel, or a silently wrong selection if you ignore it | check .shape before indexing | live |
| 20 | 05 Argo | (qc == 1) is silently all-False | decode to int first; count the codes, do not use .all() | live |
| 21 | 05 Argo | /index/, /dac/index/, _prof_index.txt all 404 | plan for screening, or use a source that has an index | live |
| 22 | 05 Argo | _meta.nc (35 KB) and _tech.nc (2.8 MB) both lack LATITUDE | position is in _prof.nc; screening costs 689 KB per candidate | live |
| 23 | 05 Argo | imports the removed _quote_string_constraints | fetch GDAC netCDF directly; do not downgrade xarray | live |
| 24 | 07 NDBC | columns named #yr, mo, dy; KeyError much later | parse lines[0], slice from lines[2] | live |
| 25 | 07 NDBC | 7,835 rows for 2019, not 8,760; file starts 5 January | resample explicitly; never assume hourly | live |
| 26 | 07 NDBC | a blanket >= 99 filter deletes real easterly winds | mask per column: 99 generally, 999 for WDIR/MWD | live |
| 27 | 07 NDBC | aligns on index; every column silently NaN | pass .to_numpy() so values are positional | live |
| 28 | 07 NDBC | mean(350, 10) = 180, the exact opposite of 0; 79.6 deg error over 2019 | average the u/v vectors, then atan2 | live |
| 29 | 08 gsw | returns nan with only a RuntimeWarning; propagates silently | pass degC, or convert with gsw.CT_from_t | live |
| 30 | 08 gsw | SA_from_SP needs pressure AND position; ~0.17 g/kg apart | use gsw.SA_from_SP(sp, p, lon, lat) | live |
| 31 | 08 gsw | a per-dbar reading of a per-1000 quantity is 100x wrong | state the units of whatever you are comparing against | live |
| 32 | 06 Copernicus | multi-minute hang on a 'describe' call | always pass --dataset-id and a filter | live |
| 33 | 06 Copernicus | not --stop-datetime, which is rejected | check --help; the CLI is not consistent across tools | live |
| 34 | 06 Copernicus | a correct dataset that looks empty | do not trust the summary; request variables explicitly | live |
| 35 | 06 Copernicus | 'Dataset not found' for GLOBAL_MULTIYEAR_PHY_001_030, which is live | the CLI wants the CMEMS product code cmems_mod_glo_phy_my_0.083deg_P1D-m | live |
| 36 | 06 Copernicus | the id looks authoritative, so you start editing a correct one | diff the failing argv against a working one before theorising | live |
| 37 | 06 Copernicus | COPERNICUS_USERNAME / COPERNICUS_PASSWORD are silently ignored | read `copernicusmarine login --help` for the real names | live |
| 38 | 09 SQL | three averages in one SELECT produce duplicate column names | alias every aggregate explicitly | live |
| 39 | 09 SQL | 31.5 Hz becomes band_32hz, 62.5 would become band_62hz | state the rule, and test the boundary case | live |

## The patterns behind them

Thirty-odd traps look like a list of accidents. They are not — they cluster into five
recurring shapes, and recognising a shape is faster than memorising a symptom.

### 1. A unit or convention mismatch that returns `nan` instead of raising
`gsw` with Kelvin. Fill values in a mean. A sentinel treated as data. Nothing errors;
everything downstream is quietly wrong. **This is the expensive class**, because the
symptom appears minutes or notebooks later, far from the cause. Defence: assert on
physical ranges at the point of computation, not at the end.

### 2. A path or name that is right in every part except one
`data/` missing. `TOL_1H` for `TOL_1h`. `line[1]` for `line[0]`. One wrong component, a
generic error message, and a strong pull towards debugging the wrong thing. Defence:
copy real values out of a real listing rather than constructing them.

### 3. A feature that exists in the examples and does nothing
`.time=first`. A no-op that returns 200 and plausible data. **The most dangerous class**,
because the absence of an error reads as confirmation. Defence: state the expected size
and row count, and check both.

### 4. Silent index alignment
`RangeIndex` into a timestamped frame. `object` dtype compared to `int`. Both return
something of the right shape, filled with the wrong thing. Defence: check `.dtype` and
`.shape` before trusting an array.

### 5. A library hiding the mechanism
`requests` and `curl` disagree about the same URL, and neither is obviously wrong,
because one of them is quietly dropping your parameter. Defence: when two tools
disagree, look at the bytes on the wire — that is what Notebook 01 is for.

## The one habit that would have prevented most of these

**State what you expect before you fetch, then assert it.** A response size, a row
count, a physical range. That is the whole discipline behind every `expect()` call in
this workshop, and it converts an afternoon of debugging into a three-second failure.

The failure it catches is not a crash. It is a `200 OK` containing a wrong answer that
looks entirely reasonable — which is the only kind of data bug that ever costs anyone a
day.

In [3]:
# The habit, in one line.
def fetch_checked(url, *, expect_bytes=None, expect_rows=None, params=None):
    """Fetch, then immediately prove the response is what you said it would be."""
    r = http.get(url, params)
    if expect_bytes is not None:
        _fetch.expect("bytes", len(r.content), expect_bytes)
    if expect_rows is not None:
        _fetch.expect("rows", len(r.text.splitlines()) - 2, expect_rows)
    return r

r = fetch_checked(S.sst_csv(point=True), expect_bytes=1324, expect_rows=30)
print("  a 1,324-byte, 30-row response. Anything else and we stop here,")
print("  before building anything on top of it.")

  !! NETWORK UNAVAILABLE -- serving the cached response
  !!   reason : offline mode (OCEAN_DATA_WORKSHOP_OFFLINE=1)
  !!   fetched: 2026-09-29 16:37
  !!   age    : 0 h
  !!   the numbers below are real data, but not necessarily current.
  ok  bytes: got 1324, expected 1324
  ok  rows: got 30, expected 30
  a 1,324-byte, 30-row response. Anything else and we stop here,
  before building anything on top of it.
